******This project retrieves the most relevant research papers using semantic search with Sentence Transformers and FAISS. Each retrieved paper is summarized using a transformer-based summarization model, analyzed with KeyBERT for keyword extraction, and processed by Gemini to identify important technical entities such as programming languages, frameworks, datasets, algorithms, and machine learning models.******

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset

In [3]:
# STEP 1 : Loaded the Research Paper Dataset

In [4]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')

In [5]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [6]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

In [7]:
# STEP 2 : Data Preprocessing
# Cleaning missing values and preparing the dataset

In [8]:
import pandas as pd

In [9]:
df=pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [10]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [11]:
df=df[['title','abstract']]

In [12]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [13]:
df.shape

(117592, 2)

In [14]:
df=df.head(15000)
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
14995,Estimation of a Low-rank Topic-Based Model for...,We consider the problem of estimating the la...
14996,A Comparison of Audio Signal Preprocessing Met...,"In this paper, we empirically investigate th..."
14997,Implicit Regularization in Deep Learning,In an attempt to better understand generaliz...
14998,A Quasi-isometric Embedding Algorithm,The Whitney embedding theorem gives an upper...


In [15]:
df.shape

(15000, 2)

In [16]:
df.isnull().sum()

title       0
abstract    0
dtype: int64

In [17]:
df["paper_text"]=df["title"]+" "+df["abstract"]
df["paper_text"].head()

C:\Users\Nishtha Goel\AppData\Local\Temp\ipykernel_7668\14272807.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["title"]+" "+df["abstract"]


0    Learning from compressed observations   The pr...
1    Sensor Networks with Random Links: Topology De...
2    The on-line shortest path problem under partia...
3    A neural network approach to ordinal regressio...
4    Parametric Learning and Monte Carlo Optimizati...
Name: paper_text, dtype: object

In [18]:
df[["paper_text"]]

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...
...,...
14995,Estimation of a Low-rank Topic-Based Model for...
14996,A Comparison of Audio Signal Preprocessing Met...
14997,Implicit Regularization in Deep Learning In ...
14998,A Quasi-isometric Embedding Algorithm The Wh...


In [19]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [20]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [21]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

In [22]:
# STEP 3 : Load Sentence Transformer
# Used to convert research papers into dense vector embeddings

In [23]:
from sentence_transformers import SentenceTransformer

In [24]:
model= SentenceTransformer("all-MiniLM-L6-v2")

In [25]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [26]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

C:\Users\Nishtha Goel\AppData\Local\Temp\ipykernel_7668\2190359946.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
C:\Users\Nishtha Goel\AppData\Local\Temp\ipykernel_7668\2190359946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.strip()


In [27]:
sample_text=df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [28]:
# STEP 4 : Generate Embeddings
# Convert every abstract into a numerical vector

In [29]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [30]:
embedding[:56]

array([-0.13156408, -0.00678266, -0.00367602,  0.03265161,  0.11219638,
        0.01227268,  0.09816723, -0.0900523 ,  0.04231162, -0.01977349,
       -0.03308421,  0.07452947,  0.10632039, -0.02060432, -0.02052104,
        0.0016949 ,  0.07081955,  0.05854451, -0.11231908,  0.02082473,
        0.05692548,  0.02015777,  0.02583104,  0.03217027,  0.10513772,
       -0.09676756,  0.02700801, -0.0234509 , -0.04549675, -0.01013699,
       -0.01794863, -0.04814427,  0.01077656, -0.03759071,  0.01943479,
        0.03715186,  0.0296784 ,  0.04330945,  0.04373207,  0.03704867,
       -0.00182592,  0.00455191, -0.00799064,  0.03037369, -0.014378  ,
        0.03795145,  0.05959162, -0.02583358, -0.06521571,  0.05900271,
       -0.02107139,  0.07359426, -0.05720102,  0.00294057,  0.00767522,
       -0.03334164], dtype=float32)

In [31]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [32]:
print(sample_embedding.shape)

(5, 384)


In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

[[1.0000002]]


In [35]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

[[0.3662528]]


In [36]:
for i in range(1,5):
    sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
    print(sim)

[[0.3662528]]
[[0.33522835]]
[[0.15505125]]
[[0.37421536]]


**Generate Full Embedding**

In [37]:
import os
import numpy as np

# 1. Define the filename where you want to store your embeddings
embedding_file = "my_embeddings.npy"

# 2. Check if you have already saved the embeddings before
if os.path.exists(embedding_file):
    print("Found saved embeddings! Loading them instantly...")
    embedding = np.load(embedding_file)
    print("Embeddings loaded. Shape:", embedding.shape)
else:
    print("Saved file not found. Running the embedding model (this may take a few minutes)...")
    
    # This is your original slow encoding line
    embedding = model.encode(
        df["paper_text"].to_list(), 
        batch_size=32, 
        show_progress_bar=True
    )
    
    # Save it immediately
    np.save(embedding_file, embedding)
    print("Encoding complete and saved to disk successfully!")

Found saved embeddings! Loading them instantly...
Embeddings loaded. Shape: (15000, 384)


In [38]:
print(embedding.shape)
print(type(embedding))

(15000, 384)
<class 'numpy.ndarray'>


In [39]:
embedding.dtype

dtype('float32')

In [40]:
# STEP 5 : Build FAISS Index
# Enables fast semantic similarity search

In [41]:
!pip install faiss-cpu

In [42]:
import faiss

In [43]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embedding)
    index = faiss.IndexFlatIP(384)
    index.add(embedding)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

Loading existing FAISS index


In [44]:
print(index.ntotal)

15000


In [45]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

(1, 384)

In [46]:
faiss.normalize_L2(query_embedding)

In [47]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

[[0.6807246  0.6709221  0.6521998  0.62811744 0.6131153 ]]
[[10466 13730 11873 12691 11282]]


In [48]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [49]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [50]:
print(df.iloc[11873]["title"])

Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [51]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    return D,I

In [52]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

[[0.6807246  0.6709221  0.6521998  0.62811744 0.6131153 ]]
[[10466 13730 11873 12691 11282]]


In [53]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score",score)
        print("Title",df.iloc[idx]["title"])
        print("Abstract",df.iloc[idx]["abstract"][:500])
        print()

In [54]:
search_paper("deep learning for medical image analysis")

Similarity Score 0.6807246
Title A Perspective on Deep Imaging
Abstract   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score 0.6709221
Title Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a 

In [55]:
!pip install transformers==4.46.3

In [56]:
# STEP 6 : Load BART Summarization Model
# Generates concise summaries of retrieved papers

In [57]:
from transformers import pipeline
summarizer=pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [58]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [59]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [60]:
type(summary)

list

In [61]:
type(summary[0])

dict

In [62]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [63]:
for score,idx in zip(D[0],I[0]):
        print("Similarity Score",score)
        print("Title",df.iloc[idx]["title"])
        print("Abstract",df.iloc[idx]["abstract"][:500])

        summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
        print(summary)
        print(summary[0]["summary_text"])
        print()

Similarity Score 0.6807246
Title A Perspective on Deep Imaging
Abstract   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or mac

In [64]:
def search_and_summarize(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score",score)
        print("Title",df.iloc[idx]["title"])
        print("Abstract",df.iloc[idx]["abstract"][:500])
        print()
        
        summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
        print(summary)
        print(summary[0]["summary_text"])
        print()

In [65]:
search_and_summarize("Deep Learning in medical imaging",k=5)

Similarity Score 0.73549867
Title A Perspective on Deep Imaging
Abstract   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or m

In [66]:
# STEP 7 : Load KeyBERT
# Extract important keywords from paper summaries

In [67]:
pip install keybert==0.8.5

Note: you may need to restart the kernel to use updated packages.


In [68]:
from keybert import KeyBERT

In [69]:
kw_model=KeyBERT(model)

In [70]:
type(kw_model)#model=SentenceTransformer("all-MiniLM-L6-v2")

keybert._model.KeyBERT

In [71]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [72]:
text=df.iloc[10466]["abstract"]
keywords=kw_model.extract_keywords(text)

In [73]:
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [74]:
print(type(keywords))

<class 'list'>


In [75]:
print(type(keywords[0]))

<class 'tuple'>


In [76]:
keywords=kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3),stop_words="english")
print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [77]:
def search_and_summarize(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score",score)
        print("Title",df.iloc[idx]["title"])
        print("Abstract",df.iloc[idx]["abstract"][:500])
        print()
        
        summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
        print(summary)
        print(summary[0]["summary_text"])
        print()
        
        keywords=kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3),stop_words="english")
        print(keywords)
        for k,s in keywords:
            print(k)

In [78]:
# STEP 8 : Configure Gemini API
# Used for LLM-based technical entity extraction

In [79]:
!pip install google-generativeai

In [88]:
from dotenv import load_dotenv 
import os 
import google.generativeai as genai 
load_dotenv() 
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") 
genai.configure(api_key=GOOGLE_API_KEY)

In [89]:
import json

In [92]:
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

In [94]:
response = gemini_model.generate_content("Reply only with the words: Gemini is working.")
print(response.text)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 33.577016281s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 33
}
]

In [ ]:
# STEP 9 : Extract Technical Named Entities
# Gemini identifies technologies, frameworks, datasets, programming languages, algorithms, and tools

In [ ]:
def extract_entities(summary):
    prompt = f"""
You are an NLP assistant.

Extract all important technical entities, technologies, machine learning concepts,
frameworks, programming languages, datasets, tools, algorithms, libraries,
organizations, and research concepts from the following research paper summary.

For every entity provide:

1. Entity Name
2. Category

Possible categories:

- Programming Language
- Framework
- Library
- Machine Learning Model
- Transformer Model
- Dataset
- Tool
- Algorithm
- Research Domain
- Healthcare Concept
- Medical Dataset
- Deep Learning Technique
- Evaluation Metric
- Organization
- Person

Return ONLY valid JSON.
Do not use Markdown.
Do not wrap the JSON inside ```json``` blocks.
Do not add any explanation.

Format:

[
  {{
    "entity": "...",
    "category": "..."
  }}
]

Text:
{summary}
"""
    response = gemini_model.generate_content(prompt)
    result = response.text.strip()
    # Remove markdown formatting if Gemini returns it
    if result.startswith("```json"):
        result = result.replace("```json", "")
        result = result.replace("```", "")
        result = result.strip()
    try:
        entities = json.loads(result)
    except Exception as e:
        print("JSON Parsing Error:", e)
        entities = []
    return entities

In [ ]:
def search_and_summarize(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    paper_no = 1
    for score,idx in zip(D[0],I[0]):
        print("="*80)
        print(f"Research Paper {paper_no}")
        print("="*80)
        print(f"Similarity Score : {score:.4f}")
        print()
        print("Title")
        print("-"*40)
        print(df.iloc[idx]["title"])
        print()
        print("Abstract")
        print("-"*40)
        print(df.iloc[idx]["abstract"][:500])
        print()
        
        summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
        summary_text = summary[0]["summary_text"]
        print("Summary")
        print("-"*40)
        print(summary_text)
        print()
        
        entities = extract_entities(summary_text)
        print("\nNamed Entities")
        print("-" * 40)
        if isinstance(entities, list):
            for item in entities:
                print(f"{item['entity']:<25} {item['category']}")
        else:
            print(entities)
        print()
        
        keywords=kw_model.extract_keywords(summary_text, keyphrase_ngram_range=(1,2),stop_words="english")
        print("Keywords")
        print("-"*40)
        for keyword,score in keywords:
            print(keyword)
        print()
        paper_no += 1

In [ ]:
search_and_summarize("machine learning in healthcare")